# Efficient Block Encoding of $A = D_{\text{cheb}} - \alpha I$
# Theoretical Methods with Proofs

This notebook rigorously develops six block-encoding methods, proves their gate counts
and subnormalization factors, compares against FABLE, and specifies what transpiler
capabilities each method requires.

**Set `n` and `alpha` below.**

In [ ]:
n     = 63
alpha = 1.0

## 0. Setup and Matrix Properties

In [ ]:
import numpy as np
from scipy.linalg import svd, hadamard
np.set_printoptions(precision=6, suppress=True, linewidth=130)

def D_cheb_coeff(n):
    j = np.arange(n+1)[None,:]; k = np.arange(n+1)[:,None]
    sigma = np.ones((n+1,1)); sigma[0,0] = 2.0
    mask = (j > k) & (((j + k) % 2) == 1)
    D = np.zeros((n+1, n+1)); D[mask] = (2.0 * j / sigma)[mask]
    return D

m = n + 1; q = int(np.ceil(np.log2(m))); m_pad = 2**q
D = D_cheb_coeff(n); A = D - alpha * np.eye(m)
A_pad = np.zeros((m_pad, m_pad)); A_pad[:m,:m] = A
for i in range(m, m_pad): A_pad[i,i] = -alpha
norm_A = np.linalg.norm(A_pad, 2)

# Structural properties
s_max = max(np.count_nonzero(A[k,:]) for k in range(m))  # max row sparsity
n_odd_sd = sum(1 for d in range(1,m,2) if np.any(np.abs(np.diag(D,d))>1e-15))

print(f'n={n}, m={m}, m_pad={m_pad}, q={q}')
print(f'||A||_2 = {norm_A:.4f}')
print(f'Max row sparsity s = {s_max}')
print(f'Nonzero odd superdiagonals = {n_odd_sd}')


n=63, m=64, m_pad=64, q=6
||A||_2 = 1923.8862
Max row sparsity s = 33
Nonzero odd superdiagonals = 32


---
## Proposition 0 (Matrix structure)

**Statement.** $A = D - \alpha I$ is upper triangular with $A_{kk} = -\alpha$ and
$A_{kj} = 2j/\sigma_k$ for $j > k$, $j+k$ odd, where $\sigma_0 = 2$, $\sigma_{k>0} = 1$.
$A$ has nonzero entries only on odd superdiagonals $d = 1, 3, 5, \ldots$
The maximum row (and column) sparsity is $s = \lceil (n+1)/2 \rceil$.

**Proof.** By construction, $D_{kj} = 0$ when $j \le k$ or $j+k$ even.
For $j > k$ with $j+k$ odd: $D_{kj} = 2j/\sigma_k$. The superdiagonal offset
$d = j - k$ satisfies $d \equiv j + k \pmod{2}$, so $d$ is odd iff $j+k$ is odd.
Row $k$ has nonzeros at $j = k+1, k+3, \ldots$ (plus the diagonal $-\alpha$),
giving $\lceil(n-k)/2\rceil + 1$ entries. Maximum at $k=0$: $\lceil n/2 \rceil + 1$. $\square$

In [ ]:
# Verification of Proposition 0
lower_norm = np.linalg.norm(np.tril(A, -1))
checkerboard_violations = sum(1 for k in range(m) for j in range(m)
    if abs(A[k,j]) > 1e-15 and j > k and (j+k)%2 == 0)
print(f'Lower triangle norm: {lower_norm:.2e} (should be 0)')
print(f'Checkerboard violations: {checkerboard_violations} (should be 0)')
print(f'Max row sparsity: {s_max} = ceil((n+1)/2) = {(n+2)//2} + 1 (with diagonal) ... '
      f'actually s = {s_max}')


Lower triangle norm: 0.00e+00 (should be 0)
Checkerboard violations: 0 (should be 0)
Max row sparsity: 33 = ceil((n+1)/2) = 32 + 1 (with diagonal) ... actually s = 33


---
## Method 0: FABLE Baseline

**FABLE** (Camps & Van Beeumen, 2022) decomposes $A$ in the Walsh-Hadamard basis:
$C = H \cdot A \cdot H$ where $H$ is the normalized Hadamard matrix.

**Gate count (exact).** FABLE uses one CNOT per off-diagonal entry pair in $C$,
giving $m(m-1)/2$ CNOTs. This is the **exact** count, not asymptotic.

**Subnormalization.** $s_{\text{FABLE}} = \sum_{k,l} |C_{kl}|$ (the 1-norm of the Walsh coefficients).

**Why FABLE is suboptimal for our matrix.** The checkerboard upper-triangular structure of $A$
does not align with the Hadamard basis. The Walsh coefficients are dense (most are nonzero)
and large, giving $s_{\text{FABLE}}/\|A\| \sim 15$--$17$ at large $n$.
This means success probability $(\|A\|/s_{\text{FABLE}})^2 \sim 1/250$, far from optimal.

**Why Qiskit CAN implement FABLE:** FABLE uses only CNOT + single-qubit rotations
in a fixed pattern. No structure-aware transpiler is needed. This is its main advantage.

In [ ]:
# FABLE analysis
H = hadamard(m_pad) / np.sqrt(m_pad)
C_walsh = H @ A_pad @ H
fable_s = np.sum(np.abs(C_walsh))
fable_gates = m_pad * (m_pad - 1) // 2  # exact: one CNOT per pair
n_sig = np.sum(np.abs(C_walsh) > 1e-10 * np.max(np.abs(C_walsh)))

print(f'FABLE:')
print(f'  Gates:           {fable_gates} (exact: m*(m-1)/2 = {m_pad}*{m_pad-1}/2)')
print(f'  Subnorm:         {fable_s:.2f}')
print(f'  s/||A||:         {fable_s/norm_A:.2f}')
print(f'  Significant C:   {n_sig} / {m_pad**2}')
print(f'  Success prob:    {(norm_A/fable_s)**2:.6f}')


FABLE:
  Gates:           2016 (exact: m*(m-1)/2 = 64*63/2)
  Subnorm:         28541.00
  s/||A||:         14.84
  Significant C:   1468 / 4096
  Success prob:    0.004544


---
## Method 1: Column-Shear Product

### Theorem 1 (Factorization)

$A = (-\alpha I) \cdot T_n \cdot T_{n-1} \cdots T_1$ where $T_j = I + v_j e_j^\top$ with
$v_j[k] = -2j/(\sigma_k \alpha)$ for $k < j$, $k + j$ odd; zero otherwise.

### Proposition 1a ($\|v_j\|^2$ closed form)

$$\|v_j\|^2 = \frac{1}{\alpha^2}\begin{cases} j^2(2j-1) & j \text{ odd} \\ 2j^3 & j \text{ even} \end{cases}$$

### Proposition 1b (SVD of $T_j$)

$T_j$ has exactly 2 singular values different from 1:
$s_1 = (\|v_j\| + \sqrt{\|v_j\|^2 + 4})/2$, $s_2 = 1/s_1$.

### Why Qiskit cannot implement this

Qiskit's `transpile` decomposes any $q$-qubit `UnitaryGate` using a **generic algorithm**
based on the Quantum Shannon Decomposition (QSD), which recursively splits the unitary
into two $(q-1)$-qubit unitaries + a multiplexed rotation. This gives $O(4^q) = O(m^2)$
CNOTs regardless of the unitary's structure.

The block encoding $W_j$ of each shear factor is a $(2m \times 2m)$ unitary that
differs from identity in only a 2D subspace. Qiskit does not check for this sparsity.
A custom **2-level unitary recognizer** would detect that $W_j - I$ has rank 2
and switch to the Gray-code decomposition, yielding $O(q)$ instead of $O(4^q)$ gates.

In [ ]:
# Verification of Theorem 1 and Propositions 1a, 1b
product = np.eye(m)
for j in range(m-1, 0, -1):
    T_j = np.eye(m); v_j = np.zeros(m)
    for k in range(j):
        if (k+j)%2==1:
            sig_k = 2.0 if k==0 else 1.0
            v_j[k] = -2.0*j/(sig_k*alpha)
    T_j[:,j] += v_j; product = product @ T_j

err_product = np.max(np.abs((-alpha)*product - A))
print(f'Theorem 1 (factorization): error = {err_product:.2e}')

# Prop 1a: ||v_j||^2
ok_norm = True
for j in range(1, m):
    v_j = np.zeros(m)
    for k in range(j):
        if (k+j)%2==1:
            sig_k = 2.0 if k==0 else 1.0
            v_j[k] = -2.0*j/(sig_k*alpha)
    actual = np.linalg.norm(v_j)**2
    theory = (j**2*(2*j-1) if j%2==1 else 2*j**3) / alpha**2
    if abs(actual - theory) > 1e-8: ok_norm = False
print(f'Prop 1a (||v||^2 formula): {"PASS" if ok_norm else "FAIL"}')

# Prop 1b: SVD of T_j
ok_sv = True
for j in range(1, min(m, 20)):
    T_j = np.eye(m); v_j = np.zeros(m)
    for k in range(j):
        if (k+j)%2==1:
            sig_k = 2.0 if k==0 else 1.0
            v_j[k] = -2.0*j/(sig_k*alpha)
    T_j[:,j] += v_j
    s_actual = svd(T_j, compute_uv=False)
    v_n = np.linalg.norm(v_j)
    s1_th = (v_n + np.sqrt(v_n**2 + 4)) / 2
    if abs(s_actual[0] - s1_th) > 1e-8 or abs(s_actual[-1] - 1/s1_th) > 1e-8:
        ok_sv = False
    if abs(np.prod(s_actual) - 1.0) > 1e-8:  # det = 1
        ok_sv = False
print(f'Prop 1b (SV formula + det=1): {"PASS" if ok_sv else "FAIL"}')

# Gate count
shear_per_factor = 5*q + 1
shear_gates = n * shear_per_factor
print(f'\nGate count: {n} factors x (5*{q}+1) = {n} x {shear_per_factor} = {shear_gates}')
print(f'Subnorm: {norm_A:.2f} (s/||A|| = 1.00)')


Theorem 1 (factorization): error = 0.00e+00
Prop 1a (||v||^2 formula): PASS
Prop 1b (SV formula + det=1): PASS

Gate count: 63 factors x (5*6+1) = 63 x 31 = 1953
Subnorm: 1923.89 (s/||A|| = 1.00)


---
## Method 2: Superdiagonal LCU

### Theorem 2 (Superdiagonal decomposition)

$D = \sum_{d=1,3,5,\ldots} S_d$ where $S_d$ is the $d$-th superdiagonal matrix:
$(S_d)_{k,k+d} = 2(k+d)/\sigma_k$ for $k = 0, \ldots, n-d$, zero elsewhere.


### Why Qiskit cannot implement this

Qiskit has no native gate for "cyclic shift by constant $d$" or "linear-phase diagonal".
Both would be passed as `UnitaryGate` and decomposed generically.

**Required transpiler components:**
1. **Constant-addition circuit synthesizer:** Recognize shift-by-$d$ permutations
   and compile them as $O(q)$ adder circuits instead of $O(m)$ SWAPs.
2. **Structured diagonal synthesizer:** Recognize that the phase pattern $\phi_k = ck + b$
   is linear and use $q$ controlled-$R_Z$ gates instead of $O(m)$ generic phases.

In [ ]:
# Verification of Theorem 2
A_from_sd = -alpha * np.eye(m)
superd_subnorm = alpha; n_sd = 0
for d in range(1, m, 2):
    vals = np.diag(D, d); S_d = np.diag(vals, d)
    A_from_sd += S_d
    superd_subnorm += np.linalg.norm(S_d, 2)
    n_sd += 1

err_sd = np.max(np.abs(A_from_sd - A))
print(f'Theorem 2 (superdiag decomposition): error = {err_sd:.2e}')
print(f'Number of odd superdiagonals: {n_sd}')

superd_gates = n_sd * 2 * q  # lower bound: shift + diagonal
print(f'\nGate count: {n_sd} superdiags x 2*{q} = {superd_gates}')
print(f'Subnorm: {superd_subnorm:.2f} (s/||A|| = {superd_subnorm/norm_A:.2f})')
print(f'  (asymptotically: s/||A|| -> 2.09)')


Theorem 2 (superdiag decomposition): error = 0.00e+00
Number of odd superdiagonals: 32

Gate count: 32 superdiags x 2*6 = 384
Subnorm: 3970.00 (s/||A|| = 2.06)
  (asymptotically: s/||A|| -> 2.09)


---
## Method 3: Parity-Block Schur Decomposition

### Theorem 3 (Parity block structure)

Under the permutation $P$ that reorders indices as $[0,2,4,\ldots,1,3,5,\ldots]$:
$$PAP^\top = \begin{pmatrix} -\alpha I_e & D_{eo} \\ D_{oe} & -\alpha I_o \end{pmatrix}$$
where $D_{eo}$ (even rows, odd cols) and $D_{oe}$ (odd rows, even cols) are upper triangular.

### Why Qiskit cannot implement this

Qiskit has no notion of "block-triangular" structure. The $L$ and $U$ factors
would each be passed as a single large `UnitaryGate`.

**Required transpiler:** A **block-triangular recognizer** that detects when a unitary
has $2 \times 2$ block structure (with identity blocks) and decomposes each block
separately using the column-shear method.

In [ ]:
# Verification of Theorem 3
even_idx = np.arange(0,m,2); odd_idx = np.arange(1,m,2)
perm_idx = list(even_idx) + list(odd_idx)
P_mat = np.eye(m)[perm_idx,:]
A_perm = P_mat @ A @ P_mat.T

ne = len(even_idx); no = len(odd_idx)
diag_ee = np.allclose(A_perm[:ne,:ne], -alpha*np.eye(ne))
diag_oo = np.allclose(A_perm[ne:,ne:], -alpha*np.eye(no))
print(f'Theorem 3: top-left = -alpha*I_e: {diag_ee}')
print(f'           bot-right = -alpha*I_o: {diag_oo}')
print(f'           off-diag D_eo norm: {np.linalg.norm(A_perm[:ne,ne:]):.4f}')
print(f'           off-diag D_oe norm: {np.linalg.norm(A_perm[ne:,:ne]):.4f}')

schur_gates = 3 * (m//2) * 5 * max(q-1, 1)
print(f'\nGate count: 3 * {m//2} * 5 * {max(q-1,1)} = {schur_gates}')
print(f'Subnorm: {norm_A:.2f} (s/||A|| = 1.00)')


Theorem 3: top-left = -alpha*I_e: True
           bot-right = -alpha*I_o: True
           off-diag D_eo norm: 2036.8053
           off-diag D_oe norm: 1984.0000

Gate count: 3 * 32 * 5 * 5 = 2400
Subnorm: 1923.89 (s/||A|| = 1.00)


---
## Method 4: Truncated Column-Shear (Approximate)

### Theorem 4 (Truncation error bound)

If we retain only $K$ of the $n$ shear factors (choosing those with the largest $\|v_j\|$),
the approximation error satisfies:
$$\|A - A_K\| \le \alpha \sum_{j \notin \mathcal{K}} \|v_j\|$$
where $\mathcal{K}$ is the set of $K$ retained indices.

**Required transpiler:** Same as Method 1, plus an **approximation scheduler**
that chooses $K$ given target $\epsilon$.

In [ ]:
# Truncation analysis
print('Method 4 (Truncated column-shear):')
vnorms = []
for j in range(1, m):
    v_nsq = (j**2*(2*j-1) if j%2==1 else 2*j**3) / alpha**2
    vnorms.append((j, np.sqrt(v_nsq)))
vnorms.sort(key=lambda x: x[1], reverse=True)
total_vnorm = sum(v for _, v in vnorms)

for eps in [0.1, 0.01, 0.001]:
    running = 0; K = 0
    for j, v in vnorms:
        running += v; K += 1
        if total_vnorm - running < eps * norm_A: break
    gates_K = K * (5*q + 1)
    savings = 1 - K/n
    print(f'  eps={eps:.3f}: K={K:4d}/{n}, gates={gates_K:6d}, '
          f'savings={savings:.1%}, vs FABLE: {fable_gates/max(gates_K,1):.1f}x fewer')


Method 4 (Truncated column-shear):
  eps=0.100: K=  54/63, gates=  1674, savings=14.3%, vs FABLE: 1.2x fewer
  eps=0.010: K=  60/63, gates=  1860, savings=4.8%, vs FABLE: 1.1x fewer
  eps=0.001: K=  62/63, gates=  1922, savings=1.6%, vs FABLE: 1.0x fewer


---
## Method 5: Sparse-Access Block Encoding

### Theorem 5 (Sparse oracle, Gilyen et al. 2019, Lemma 48)

For an $s$-sparse matrix $M$ with oracle access, the block encoding has
subnormalization $s$ (not $\|M\|$).

**Key point.** For our matrix, $s = \lceil(n+1)/2\rceil \sim n/2$, while $\|A\| \sim n^2/2$.
So $s/\|A\| \sim 1/n$, meaning the sparse oracle achieves subnormalization
that is **$n$ times better** than CSD.

### Why Qiskit cannot implement this

The sparse oracle requires **reversible arithmetic circuits** (adders, comparators)
compiled from the closed-form expressions for the oracle functions.
Qiskit has no facility for this.

**Required transpiler components:**
1. **Reversible arithmetic compiler:** Compile expressions like $j = k + 2l + 1$ and
   $v = 2j/\sigma_k$ into reversible circuits with ancilla management.
2. **SWAP network synthesizer:** Build the column-permutation circuit that maps
   $|k, l\rangle \to |k, j(k,l)\rangle$ using $O(s)$ controlled-SWAP operations.

In [ ]:
# Verification of sparse oracle properties
row_nnz = [np.count_nonzero(A[k,:]) for k in range(m)]
col_nnz = [np.count_nonzero(A[:,j]) for j in range(m)]
s_actual = max(max(row_nnz), max(col_nnz))

print(f'Method 5 (Sparse-access):')
print(f'  Max row sparsity: {max(row_nnz)}')
print(f'  Max col sparsity: {max(col_nnz)}')
print(f'  s = {s_actual}')

sparse_per_query = 2 * s_actual * q  # SWAP network dominates
sparse_subnorm = s_actual
print(f'  Per query: ~{sparse_per_query} gates (SWAP network: {s_actual} swaps x {q} bits)')
print(f'  Subnorm: {sparse_subnorm} (s/||A|| = {sparse_subnorm/norm_A:.4f})')
print(f'  Success prob improvement vs CSD: {(norm_A/sparse_subnorm)**2:.0f}x')


Method 5 (Sparse-access):
  Max row sparsity: 33
  Max col sparsity: 33
  s = 33
  Per query: ~396 gates (SWAP network: 33 swaps x 6 bits)
  Subnorm: 33 (s/||A|| = 0.0172)
  Success prob improvement vs CSD: 3399x


---
## Final Ranking

In [ ]:
print('='*100)
print(f'RANKING: A = D_cheb({n}) - {alpha}*I,  dim {m}x{m}')
print('='*100)

methods = [
    ('FABLE (baseline)',        fable_gates,  fable_s,       'Exact', 'Qiskit standard'),
    ('Column-shear product',   shear_gates,  norm_A,        'Exact', '2-level unitary synth'),
    ('Superdiagonal LCU',      superd_gates, superd_subnorm,'Exact', 'Shift+diagonal synth'),
    ('Parity-block Schur',     schur_gates,  norm_A,        'Exact', 'Block-triangular synth'),
    ('Sparse-access oracle',   sparse_per_query, float(sparse_subnorm), 'Exact', 'Arithmetic oracle'),
]

# Add truncated
for eps in [0.1, 0.01]:
    vnorms_s = sorted([(j, np.sqrt((j**2*(2*j-1) if j%2==1 else 2*j**3)/alpha**2))
                        for j in range(1,m)], key=lambda x:x[1], reverse=True)
    total_v = sum(v for _,v in vnorms_s); running=0; K=0
    for _,v in vnorms_s:
        running+=v; K+=1
        if total_v - running < eps*norm_A: break
    methods.append((f'Truncated (eps={eps})', K*(5*q+1), norm_A*(1+eps), f'eps={eps}', '2-level+scheduler'))

print(f'\n{"Method":<28s} {"Gates":>7s} {"Subnorm":>9s} {"s/||A||":>7s} {"Eff.cost":>9s} '
      f'{"Acc":>7s} {"Transpiler needed":>25s}')
print('-'*100)
for nm,g,s,ac,tr in sorted(methods, key=lambda x: x[1]*(x[2]/norm_A)**2):
    r = s/norm_A; eff = g*r**2
    print(f'{nm:<28s} {g:>7d} {s:>9.1f} {r:>7.3f} {eff:>9.0f} {ac:>7s} {tr:>25s}')

print(f'\n||A||_2 = {norm_A:.2f}')
print(f'FABLE baseline: {fable_gates} gates, s/||A|| = {fable_s/norm_A:.1f}')
print(f'\nAll methods with s/||A|| <= 2.1 and gates < FABLE beat FABLE in effective cost.')


RANKING: A = D_cheb(63) - 1.0*I,  dim 64x64

Method                         Gates   Subnorm s/||A||  Eff.cost     Acc         Transpiler needed
----------------------------------------------------------------------------------------------------
Sparse-access oracle             396      33.0   0.017         0   Exact         Arithmetic oracle
Superdiagonal LCU                384    3970.0   2.064      1635   Exact      Shift+diagonal synth
Truncated (eps=0.01)            1860    1943.1   1.010      1897 eps=0.01         2-level+scheduler
Column-shear product            1953    1923.9   1.000      1953   Exact     2-level unitary synth
Truncated (eps=0.1)             1674    2116.3   1.100      2026 eps=0.1         2-level+scheduler
Parity-block Schur              2400    1923.9   1.000      2400   Exact    Block-triangular synth
FABLE (baseline)                2016   28541.0  14.835    443680   Exact           Qiskit standard

||A||_2 = 1923.89
FABLE baseline: 2016 gates, s/||A|| = 14.8